# 07. Spectral indices without shortcuts

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/07-imagery.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=07-imagery.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/07-imagery.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `f6ba8cdc16b2c7cf3cfa640b0e13bcc74e8cde09eb115b74a01c03fd1a991442`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 7: Spectral indices without shortcuts; abstract contour illustration](assets/07-imagery.svg)

# 07. Spectral indices without shortcuts

Synthetic reflectance and NDVI

**Learning goals:** Compute NDVI with TerrIO; handle zero denominators; inspect threshold sensitivity and quality masks.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=07-imagery.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## Reflectance has a provenance

These bands are invented unitless reflectances, not a satellite acquisition. We use them to isolate arithmetic and quality-control choices. Real imagery needs the provider's scale/offset, cloud flags, acquisition time, spatial resolution, and product license.

In [ ]:
y, x = np.mgrid[0:1:96j, 0:1:96j]
red = (.08 + .22*x).astype("float32")
nir = (.15 + .50*y).astype("float32")
red[:8, :8] = 0; nir[:8, :8] = 0
red_path, nir_path = OUTPUT / "red.tif", OUTPUT / "nir.tif"
raster(red).write(str(red_path)); raster(nir).write(str(nir_path))
image_grid([red, nir], ["Synthetic red", "Synthetic NIR"], ["Reds", "Purples"], units="reflectance")
plt.show()

## Normalize with an explicit validity mask

NDVI = (NIR − red)/(NIR + red). The index is undefined when the denominator is zero. We replace invalid band pixels with nodata before asking TerrIO to calculate, and verify valid values against independent NumPy arithmetic.

In [ ]:
valid = (red + nir) > 0
for band, path in [(red, red_path), (nir, nir_path)]:
    raster(np.where(valid, band, -9999)).write(str(path))
ndvi_raster = terrio.calc("(B - A) / (B + A)", A=str(red_path), B=str(nir_path), type="Float32")
ndvi = ndvi_raster.read(1)
reference = np.full(red.shape, np.nan)
np.divide(nir-red, nir+red, out=reference, where=valid)
np.testing.assert_allclose(ndvi[valid], reference[valid], atol=1e-6)
image_grid([np.where(valid, ndvi, np.nan)], ["NDVI on valid pixels"], ["RdYlGn"], units="index")
plt.show()

## A threshold is a choice

A higher NDVI can be consistent with vegetation, but the index alone is not a land-cover classifier, health diagnosis, or biomass estimate. The graph asks how the reported fraction changes with the chosen threshold.

In [ ]:
thresholds = np.linspace(0, .8, 41)
fractions = [(ndvi[valid] > t).mean() for t in thresholds]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(thresholds, fractions)
ax.set(xlabel="NDVI threshold", ylabel="Fraction of valid pixels above threshold", ylim=(0, 1), title="Synthetic threshold sensitivity")
plt.show()
assert all(a >= b for a, b in zip(fractions, fractions[1:]))

## Separate quality masks from scientific thresholds

An illustrative cloud strip removes samples before summarizing. This mask is synthetic; no cloud detection is being performed. Different masks can alter a mean even when the formula is unchanged.

In [ ]:
clear = valid.copy(); clear[:, 30:45] = False
print("All valid mean:", float(ndvi[valid].mean()))
print("Synthetic clear-sky mean:", float(ndvi[clear].mean()))
print("Excluded fraction:", 1-clear.sum()/valid.sum())

## Your investigation

Move the synthetic cloud strip. Explain why comparing dates with different missing areas can create a misleading apparent change.

### Discussion guide

The comparison can reflect different sampled locations. Compare a common valid footprint and report coverage as well as the index.

## Sources and credit

- [Source 1](https://terrio.dev/python/)
- [Source 2](https://www.usgs.gov/landsat-missions/landsat-normalized-difference-vegetation-index)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.